## Adaptation of Hunstville Ignition Energy Paper for Turbopump TCA

In [5]:
import cantera as ct
import CoolProp as cp
import numpy as np
import pandas as pd

In [6]:
PSI2PA = 6894.76 # psi to pa
R2K = 0.555556 # Rankine to Kelvin
BTU2J = 1055.06 # BTU to J
LBM2KG = 0.453592 # lbm to kg

In [7]:
# Torch
fuel_torch = "hydrogen"
ox_torch = "oxygen"
mech_torch = "h2_sandiego.yaml" # torch reaction mechanism (H2)
OF_torch = 1
pc_torch = 300 * PSI2PA

# MCA
fuel = "JetA"
ox = "oxygen"
mech_MCA = "A2NTC_skeletal.yaml" # MCA reaction mechanism (Jet-A)
OF_MCA = 2.2
mdot_main = 20 * LBM2KG


T1_f = 298
T2_f = 500
T1_ox = 90
T2_ox = 500
Pc = 500 * PSI2PA


### Find Change in enthalpy required to reach Autoigniton Temperature for MCA Propellants

In [8]:
h1_ox = cp.CoolProp.PropsSI("H", "T", T1_ox, "P", Pc, ox)
h2_ox = cp.CoolProp.PropsSI("H", "T", T2_ox, "P", Pc, ox)
h1_f = cp.CoolProp.PropsSI("H", "T", T1_f, "P", Pc, fuel)
h2_f = cp.CoolProp.PropsSI("H", "T", T2_f, "P", Pc, fuel)

dh_f = h2_f - h1_f
dh_ox = h2_ox - h1_ox

df = pd.DataFrame([dh_f*1e-6, dh_ox*1e-6], [fuel, ox], columns=["Delta h [MJ/kg]"])

dh_mix = (1/(OF_MCA+1)) * dh_f + (OF_MCA/(OF_MCA+1)) * dh_ox # J/kg
print("MCA Propellants: \n State 1: Injection \n State 2: Autoignition Temperature\n")
print(df)
print(f"\nMass Averaged dh: {dh_mix * 10**-6:0.3f} [MJ/kg]")

ValueError: Initialize failed for backend: "?", fluid: "JetA" fractions "[ 1.0000000000 ]"; error: key [JetA] was not found in string_to_index_map in JSONFluidLibrary : PropsSI("H","T",298,"P",3447380,"JetA")

In [222]:
# function to get masss averaged chemical potential of mixture 
def mass_avg_chem_pot(mixture):
    mu = mixture.chemical_potentials      # J/kmol
    W  = mixture.molecular_weights        # kg/kmol
    Y  = mixture.Y                        # mass fractions
    return np.dot(Y, mu / W)              # J/kg mixture

In [223]:
# find difference in chemical potentials in torch reaction to find energy released from reaction
torch_rxn = ct.Solution(mech_torch)
torch_rxn.TPY = 298, pc_torch, {"O2": OF_torch/(OF_torch+1), "H2":(1/(OF_torch+1)) } 

# get starting chemical potentials, equillibriate reaction and find difference
pre_rxn = mass_avg_chem_pot(torch_rxn)
torch_rxn.equilibrate("HP")
post_rxn = mass_avg_chem_pot(torch_rxn)

combustion_energy_torch = np.abs(post_rxn - pre_rxn)


print("Torch Energy Release:\n")
print(f"Flame Temp: {torch_rxn.T:0.1f} [K]")
print(f"Combustion Energy: {combustion_energy_torch* 10**-6:0.3f} [MJ/kg]")

Torch Energy Release:

Flame Temp: 1282.0 [K]
Combustion Energy: 42.307 [MJ/kg]


In [224]:
# same process for MCA Combustion energy
# find difference in chemical potentials in torch reaction to find energy released from reaction
MCA_rxn = ct.Solution(mech_MCA)
MCA_rxn.TPY = T2_f, Pc, {"O2": OF_MCA/(OF_MCA+1), "POSF10325":(1/(OF_MCA+1)) } 

# get starting chemical potentials, equillibriate reaction and find difference
pre_rxn = mass_avg_chem_pot(MCA_rxn)
MCA_rxn.equilibrate("HP")
post_rxn = mass_avg_chem_pot(MCA_rxn)

combustion_energy_MCA = np.abs(post_rxn - pre_rxn)

print("MCA Energy Release:\n")
print(f"Flame Temp: {MCA_rxn.T:0.1f} [K]")
print(f"Combustion Energy: {combustion_energy_MCA* 10**-6:0.3f} [MJ/kg]")

MCA Energy Release:

Flame Temp: 3528.1 [K]
Combustion Energy: 40.114 [MJ/kg]


In [225]:
# Compute percent of MCA that needs to ignite for chain reaction
x = dh_mix/combustion_energy_MCA 
P_ig = mdot_main * x * dh_mix

print(f"{x*100:0.3f}% of mdot_main to cause chain reaction")
print(f"Torch Power Required: {P_ig*10**-3:0.3f} [kW]")

1.420% of mdot_main to cause chain reaction
Torch Power Required: 73.341 [kW]


In [227]:
nu = 0.2 # Heat Transfer Efficiency from torch exhaust to MCA Propellants
mdot_torch = P_ig / (combustion_energy_torch * nu)
mdot_torch_perfect = P_ig / (combustion_energy_torch)


print(f"Assuming Perfect Heat Transfer:")
print(f" mdot Torch: {mdot_torch_perfect:0.5f} [kg/s]\n")

print(f"Assuming Heat Transfer Efficiency of {nu*100:0.2f}%")
print(f" mdot Torch: {mdot_torch:0.5f} [kg/s]")
print(f" mdot Torch: {mdot_torch/LBM2KG:0.5f} [lbm/s]")



Assuming Perfect Heat Transfer:
 mdot Torch: 0.00173 [kg/s]

Assuming Heat Transfer Efficiency of 20.00%
 mdot Torch: 0.00867 [kg/s]
 mdot Torch: 0.01911 [lbm/s]


### Tabled for now: Injecting pilot flame into a wsr
#### -> can be either (likely both) torch combustion products -> torch ignition or torch combustion products -> MCA 
#### -> may be able to incorporate flame kernel radius into this